# 05b_finetune_mlx: Fine-Tune on Apple Silicon (MLX variant of notebook 05)

**Session:** Day 2, S11 — Lab: fine-tune it. This is the **learner variant** for a participant on an Apple Silicon Mac (M1 or later). Same lab, same decisions, same output as `05_finetune`. **It does not run in Colab** — in Colab, use `05_finetune`.
**Expected runtime:** training is **budgeted at 25 minutes**, checked between epochs. **Not yet measured on a Mac** — this notebook was built and executed end to end on MLX's Linux CPU backend, which proves every step but says nothing about Apple Silicon speed. Time it on the first Mac that runs it and record the figure in `docs/timing_log.md`.
**Needs:** macOS on Apple Silicon, Python 3.11 or 3.12, about 8 GB of free memory. No API key, no Hugging Face account, no GPU runtime. Installs `requirements-mlx.txt` (three pinned packages) if missing. Reads your `train_clean.jsonl` / `val_clean.jsonl` from notebook 04; rebuilds them from `data/finetune/` if they are not there. Downloads about 2.5 GB of model weights.
**A correct result looks like:** the training loss falls from about **0.7** to below **0.1**, the validation loss falls after every epoch, and the five preview tickets come back as **5 of 5 schema-valid JSON records** with `asset_tag` `null` where the ticket names no asset. The final cell prints `ADAPTER READY` — and the adapter it names is in **the same PEFT format notebook 05 produces**, so notebook 06, `scripts/register_adapter.py` and the `tuned` endpoint work on it unchanged.

> All data in this lab is synthetic. No real OQ material anywhere.

---
**The plan.** Identical to notebook 05: load the cleaned dataset → look at what the model reads → load the base model → ask five tickets *before* → choose the adapter (**TODO 1**) → choose the training settings (**TODO 2**) → train → read the loss → load back → ask the same five tickets *after*. One extra step at the end: **convert** the MLX adapter into the PEFT format.

**What is different from notebook 05, honestly.** (1) MLX instead of PyTorch — Apple's array library, which uses the Mac's GPU and shared memory. (2) The base model is loaded in 16-bit, not 4-bit: a 1B model fits easily, so this is LoRA rather than QLoRA. (3) Checkpoints are **per epoch**, not every ten steps: mlx-lm can reload adapter weights but keeps no record of how far it got, so this notebook trains one epoch at a time and writes progress after each. A crash costs at most one epoch. (4) The optimizer starts fresh after a crash (its momentum is not saved); in an uninterrupted run it is kept across epochs.

**No Mac GPU today?** Set `SMOKE_TEST = True`: a tiny model on four rows. It proves the plumbing; it does not learn the task.

**Why this cell:** the same first cell as every notebook in the repo. On a Mac it takes the *local* branch: it finds the repo root and points `CHECKPOINT_DIR` at `checkpoints/local/`. There is no Drive to mount — your disk does not disconnect — but the checkpoint habit is the same, because lids close and kernels die.

In [ ]:
# Environment detection: Colab vs local. Sets IN_COLAB, REPO_ROOT, CHECKPOINT_DIR.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# The repo URL participants clone in Colab. Set once, here.
REPO_URL = "https://github.com/Utkarsh-09/AI_GURU_labs.git"

if IN_COLAB:
    # Drive first: checkpoints survive a runtime disconnect.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_ROOT = Path("/content/oq-advanced-ai")
    if not REPO_ROOT.exists():
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/oq-advanced-ai-checkpoints")
else:
    # Local: find the repo root by walking up until BUILD_SPEC.md appears.
    here = Path.cwd()
    REPO_ROOT = next(
        (p for p in [here, *here.parents] if (p / "BUILD_SPEC.md").exists()),
        here,
    )
    CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / "local"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Make repo modules importable: config.endpoints, notebooks/utils.py
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

print(f"Environment : {'Colab' if IN_COLAB else 'local'}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Checkpoints : {CHECKPOINT_DIR}")

**Why this cell:** MLX is not part of the base requirements, because it only runs on Apple hardware. The pins match `requirements-mlx.txt` exactly. If `mlx-lm` is already installed the cell installs nothing.

In [ ]:
# Pinned installs — versions match requirements-mlx.txt. Apple Silicon only.
import importlib.util

if IN_COLAB:
    raise SystemExit("This is the Apple Silicon variant. In Colab, open 05_finetune.ipynb instead.")
if importlib.util.find_spec("mlx_lm") is None:
    %pip install -q mlx==0.32.2 mlx-lm==0.31.3 transformers==5.16.1 safetensors==0.8.0
print("Install cell done.")

**Why this cell:** every choice that changes *what gets trained*, in one place. Same names as notebook 05. `RUN_NAME` names the folder where this run's adapter and progress live — change it to keep a second experiment next to the first.

In [ ]:
import mlx.core as mx

import finetune_mlx_utils
import finetune_utils

MODEL_NAME = "llama3.2-1b"       # see finetune_utils.MODEL_CHOICES
RUN_NAME = "run1"                # a new name = a new, separate experiment
SMOKE_TEST = False               # True = tiny model, four rows: a plumbing test

TRAINING_BUDGET_MINUTES = 25     # checked between epochs: no new epoch starts past it
MAX_LENGTH = 1024                # longest row allowed, in tokens
SEED = 42

# Facilitator switch: lets an automated test flip the smoke test on.
if os.environ.get("OQ_SMOKE_TEST") == "1":
    SMOKE_TEST = True
if SMOKE_TEST:
    MODEL_NAME = "smoke-test"

mx.random.seed(SEED)
model_choice = finetune_utils.MODEL_CHOICES[MODEL_NAME]
run_dir = CHECKPOINT_DIR / "05b_finetune_mlx" / f"{MODEL_NAME}_{RUN_NAME}"

print(f"model      : {MODEL_NAME}  ({model_choice['mlx_repo']})")
print(f"run folder : {run_dir}")
print(f"MLX device : {mx.default_device()}   (gpu = Apple Silicon; cpu = the slow test path)")

**Why this cell:** the same data as notebook 05 — the dataset *you* cleaned in notebook 04 (373 training rows, 72 validation rows), or the same thing rebuilt from the committed files if notebook 04's output is not on this machine. The 20 held-out tickets are only ever *asked*, never trained on.

In [ ]:
import dataset_utils

train_pairs, val_pairs, data_source = finetune_utils.load_clean_pairs(REPO_ROOT, CHECKPOINT_DIR)
heldout_pairs = dataset_utils.load_jsonl(REPO_ROOT / "data" / "eval" / "heldout_20.jsonl")
schema = dataset_utils.load_schema(REPO_ROOT / "data" / "finetune" / "ticket_schema.json")

if SMOKE_TEST:
    train_pairs = train_pairs[:finetune_mlx_utils.SMOKE_TEST_TRAIN_ROWS]
    val_pairs = val_pairs[:finetune_mlx_utils.SMOKE_TEST_VAL_ROWS]

print(f"source   : {data_source}")
print(f"train    : {len(train_pairs)} rows")
print(f"val      : {len(val_pairs)} rows")
print(f"held-out : {len(heldout_pairs)} rows (for asking, never for training)")

**Why this cell:** load the base model and its tokenizer. The weights stay frozen for the whole lab. The helper also swaps the tokenizer's chat template for the text **Ollama** sends, because that is who will serve your tuned model: Hugging Face's template for this model adds a `Today Date:` line that Ollama's does not, and a model trained on a header it never sees again is a model trained on noise.

In [ ]:
model, tokenizer, template_note = finetune_mlx_utils.load_model_and_tokenizer(model_choice["mlx_repo"])

_, base_parameter_count = finetune_mlx_utils.count_parameters(model)
print(f"loaded     : {model_choice['mlx_repo']}")
print(f"parameters : {base_parameter_count / 1e6:,.0f} M")
print(template_note)

**Why this cell:** one full training row exactly as the model reads it — a single string with marker tokens between the roles. Whatever is in the last block is what the model learns to write.

In [ ]:
example_pair = train_pairs[0]
rendered_text = tokenizer.apply_chat_template(example_pair["messages"], tokenize=False)
print(rendered_text[:330])
print("   ...")
print(rendered_text[-300:])

**Why this cell:** training cost is counted in tokens. Each row becomes token ids plus the position where the answer starts; the loss is computed **only on the answer** (`mask_prompt`), the same rule notebook 05 applies by hand. Otherwise most of the training signal would go into memorising the system prompt.

In [ ]:
train_set = finetune_mlx_utils.build_chat_dataset(train_pairs, tokenizer)
val_set = finetune_mlx_utils.build_chat_dataset(val_pairs, tokenizer)

train_tokens, train_graded_tokens = finetune_mlx_utils.count_tokens(train_set)
longest_row = max(len(train_set[position][0]) for position in range(len(train_set)))

print(f"train rows            : {len(train_set)}")
print(f"tokens per epoch      : {train_tokens:,}")
print(f"  of which are graded : {train_graded_tokens:,}  ({train_graded_tokens / train_tokens:.0%} - the JSON answers)")
print(f"longest row           : {longest_row} tokens (limit {MAX_LENGTH})")
assert longest_row <= MAX_LENGTH, "a row is longer than MAX_LENGTH and would be cut off - raise MAX_LENGTH"

**Why this cell (milestone 1):** measure before you change anything. Five held-out tickets go to the *untuned* model and each reply is checked strictly: does `json.loads` accept it as it stands, and does it pass the ticket schema? Expect the score to look fine — a modern small model can already produce valid JSON. **Read the replies.** Look at `asset_tag` (is that tag anywhere in the ticket, or is it the example from the system prompt?) and at `routing_queue`. Saved, so a re-run loads instead of asking again.

In [ ]:
import utils

PREVIEW_COUNT = finetune_mlx_utils.SMOKE_TEST_PREVIEW_COUNT if SMOKE_TEST else 5
NEW_TOKENS = finetune_mlx_utils.SMOKE_TEST_NEW_TOKENS if SMOKE_TEST else 160
preview_pairs = heldout_pairs[:PREVIEW_COUNT]

before_replies = utils.load_json(run_dir, "replies_before", default=None)
if before_replies is None:
    before_replies = []
    for pair in preview_pairs:
        reply_text = finetune_mlx_utils.generate_reply(model, tokenizer, pair["messages"][:2], NEW_TOKENS)
        before_replies.append({"ticket_id": pair["ticket_id"], "reply": reply_text})
    utils.save_json(run_dir, "replies_before", before_replies)

before_valid_count = 0
for item in before_replies:
    verdict = finetune_utils.check_reply(item["reply"], schema)
    before_valid_count += verdict["valid"]
    print(f"{item['ticket_id']}  schema-valid: {verdict['valid']}")
    print(f"  {item['reply'][:300]}")
print()
print(f"BEFORE training: {before_valid_count} of {len(before_replies)} replies are schema-valid")

**Why this cell — TODO 1, the adapter:** the same four decisions as notebook 05, with the same names, so a group on a Mac and a group on Colab can compare notes.

- `LORA_RANK` — width of the two small matrices trained beside each layer. 8 to 32 is normal; a *format* task needs little.
- `LORA_ALPHA` — how loudly the adapter speaks: its output is scaled by `alpha / rank`. (mlx-lm takes that ratio directly and calls it `scale`; the helper does the division.)
- `LORA_DROPOUT` — regularisation. With 373 rows, 0.05 is reasonable.
- `TARGET_MODULES` — which layers get an adapter: attention only (`q_proj k_proj v_proj o_proj`) is lighter; adding the MLP layers (`gate_proj up_proj down_proj`) is what matters most for quality.

In [ ]:
# ── TODO 1 ─────────────────────────────────────────────────────────
# Choose the adapter. Hint: start with rank 16, alpha 32, dropout 0.05 and
# all seven layer names from the text above (a Python list of strings).
LORA_RANK = ...        # <- replace the ... with your choice
LORA_ALPHA = ...       # <- replace the ... with your choice
LORA_DROPOUT = ...     # <- replace the ... with your choice
TARGET_MODULES = ...   # <- replace the ... with your choice
# ───────────────────────────────────────────────────────────────────

assert LORA_RANK is not ... and LORA_ALPHA is not ..., "TODO 1 is not filled in yet"
assert LORA_DROPOUT is not ... and TARGET_MODULES is not ..., "TODO 1 is not filled in yet"

lora_parameters = finetune_mlx_utils.build_lora_parameters(LORA_RANK, LORA_ALPHA, LORA_DROPOUT, TARGET_MODULES)
finetune_mlx_utils.add_lora_layers(model, lora_parameters)

trainable_count, total_count = finetune_mlx_utils.count_parameters(model)
print(f"rank {LORA_RANK}, alpha {LORA_ALPHA} (mlx-lm scale {lora_parameters['scale']:.1f}), dropout {LORA_DROPOUT}")
print(f"adapters on : {', '.join(lora_parameters['keys'])}")
print(f"training    : {trainable_count:,} of {total_count:,} parameters ({trainable_count / total_count:.2%})")

**Why this cell — TODO 2, the training settings:** the same four numbers as notebook 05.

- `NUM_EPOCHS` — passes over the rows. Small datasets usually want 2 to 4. **This is the one that spends your 25 minutes.**
- `LEARNING_RATE` — LoRA wants far more than full fine-tuning: `2e-4` is the usual start.
- `BATCH_SIZE` — rows per forward pass, limited by memory. 4 is safe.
- `GRADIENT_ACCUMULATION` — batches added up before one update. `BATCH_SIZE × GRADIENT_ACCUMULATION` is the *effective* batch; 16 is a sensible target.

mlx-lm counts **iterations** (one batch each), not updates, so the log below is in iterations.

In [ ]:
# ── TODO 2 ─────────────────────────────────────────────────────────
# Choose the training settings. Hint: 3 epochs, learning rate 2e-4,
# batch size 4, gradient accumulation 4 (effective batch 16).
NUM_EPOCHS = ...              # <- replace the ... with your choice
LEARNING_RATE = ...           # <- replace the ... with your choice
BATCH_SIZE = ...              # <- replace the ... with your choice
GRADIENT_ACCUMULATION = ...   # <- replace the ... with your choice
# ───────────────────────────────────────────────────────────────────

assert NUM_EPOCHS is not ... and LEARNING_RATE is not ..., "TODO 2 is not filled in yet"
assert BATCH_SIZE is not ... and GRADIENT_ACCUMULATION is not ..., "TODO 2 is not filled in yet"

if SMOKE_TEST:
    BATCH_SIZE = 2                # four rows cannot fill a batch of 4 twice
    GRADIENT_ACCUMULATION = 1

iterations_per_epoch = len(train_set) // BATCH_SIZE
print(f"effective batch : {BATCH_SIZE * GRADIENT_ACCUMULATION} rows per update")
print(f"iterations      : {iterations_per_epoch} per epoch x {NUM_EPOCHS} epochs = {iterations_per_epoch * NUM_EPOCHS}")
print(f"tokens to read  : {train_tokens * NUM_EPOCHS:,}")

**Why this cell:** checkpoints are only useful if they belong to *this* experiment. The helper writes your settings into the run folder the first time and compares them on every later visit: same settings → `resume` or `finished`; different settings → it stops and asks for a new `RUN_NAME`. If epochs were already completed, the saved adapter weights go back into the model here, and any half-logged epoch is dropped from the loss log because it is about to be redone.

In [ ]:
run_settings = {
    "model": model_choice["mlx_repo"],
    "smoke_test": SMOKE_TEST,
    "train_rows": len(train_set),
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "lora_parameters": lora_parameters,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
}
run_status = finetune_utils.check_run_folder(run_dir, run_settings)
progress = finetune_mlx_utils.load_progress(run_dir)

if progress["epochs_done"] > 0:
    finetune_mlx_utils.restore_adapter_weights(model, run_dir)
    finetune_mlx_utils.drop_unfinished_epoch_from_log(run_dir, progress)

print(f"run folder  : {run_dir}")
print(f"status      : {run_status}")
print(f"epochs done : {progress['epochs_done']} of {NUM_EPOCHS}"
      f"{'  (adapter weights restored from disk)' if progress['epochs_done'] > 0 else ''}")

**Why this cell (milestone 2) — the training:** one epoch per turn of the loop. After each epoch: measure the validation loss, save the adapter, write the progress file. That order is what makes a crash cheap — re-running this cell continues from the last finished epoch, and does nothing at all if every epoch is done. The time budget is checked *between* epochs: no new epoch starts once the budget is spent. Watch the `loss` column fall fast at first (the model finds the format), then slowly (it learns your conventions).

In [ ]:
budget_seconds = TRAINING_BUDGET_MINUTES * 60
optimizer = finetune_mlx_utils.make_optimizer(LEARNING_RATE)
REPORT_EVERY = 1 if SMOKE_TEST else 8

if progress["epochs_done"] >= NUM_EPOCHS:
    print("Every epoch of this run is already done. Nothing to train.")

while progress["epochs_done"] < NUM_EPOCHS:
    if progress["training_seconds"] >= budget_seconds:
        print(f"TIME BUDGET reached after {progress['epochs_done']} epochs: keeping what has been learned so far.")
        break
    epoch_seconds = finetune_mlx_utils.train_one_epoch(
        model, optimizer, train_set, run_dir, progress,
        BATCH_SIZE, GRADIENT_ACCUMULATION, MAX_LENGTH, REPORT_EVERY)
    eval_loss = finetune_mlx_utils.validation_loss(model, val_set, BATCH_SIZE, MAX_LENGTH)
    finetune_mlx_utils.save_mlx_adapter(model, run_dir, model_choice["mlx_repo"], lora_parameters)
    progress = finetune_mlx_utils.record_epoch(run_dir, progress, epoch_seconds, iterations_per_epoch, eval_loss)
    print(f"   epoch {progress['epochs_done']} done: validation loss {eval_loss:.4f}, adapter and progress saved")

print()
print(f"epochs done: {progress['epochs_done']} of {NUM_EPOCHS}, {progress['training_seconds'] / 60:.1f} minutes of training")

**Why this cell:** the loss log was written to the run folder as training went, so the table is complete even if the run happened in two sittings. **Training loss:** did it fall and flatten, or is it still dropping (more epochs would help) or bouncing (learning rate too high)? **Validation loss**, once per epoch, on tickets the model never trained on: if it starts *rising* while training loss keeps falling, the model has begun memorising.

In [ ]:
loss_log = finetune_utils.load_loss_log(run_dir)
train_entries = [entry for entry in loss_log if "loss" in entry]
eval_entries = [entry for entry in loss_log if "eval_loss" in entry]

print("iteration   epoch   train loss   minutes")
show_every = max(len(train_entries) // 12, 1)
for position, entry in enumerate(train_entries):
    is_last = position == len(train_entries) - 1
    if position % show_every == 0 or is_last:
        print(f"{entry['step']:>9}   {entry['epoch']:>5.2f}   {entry['loss']:>10.4f}   {entry['minutes']:>7.1f}")

print()
print("validation loss at the end of each epoch:")
for entry in eval_entries:
    print(f"  epoch {entry['epoch']:>4.1f}   eval loss {entry['eval_loss']:.4f}")

first_loss = train_entries[0]["loss"]
last_loss = train_entries[-1]["loss"]
training_minutes = progress["training_seconds"] / 60
print()
print(f"training loss {first_loss:.3f} -> {last_loss:.3f} in {training_minutes:.1f} minutes of training")

**Why this cell (milestone 3) — same adapter format as notebook 05:** MLX saved the adapter its own way. Everything downstream — notebook 06, `scripts/register_adapter.py`, Ollama — expects the Hugging Face PEFT layout. The numbers are the same; only the packaging differs: both matrices are transposed, and PEFT's `alpha` is set to `scale × rank` so that `alpha / rank` gives back mlx-lm's `scale`. The converted adapter is saved twice, like notebook 05: in the run folder and under `checkpoints/` in the repo.

In [ ]:
import shutil

mlx_adapter_dir = run_dir / finetune_mlx_utils.MLX_ADAPTER_FOLDER
final_adapter_dir = run_dir / finetune_utils.FINAL_ADAPTER_FOLDER
repo_adapter_dir = REPO_ROOT / "checkpoints" / f"adapter_{MODEL_NAME}_{RUN_NAME}_mlx"

conversion = finetune_utils.convert_mlx_adapter_to_peft(mlx_adapter_dir, final_adapter_dir, model_choice["hf_repo"])
tokenizer.save_pretrained(str(final_adapter_dir))
shutil.copytree(final_adapter_dir, repo_adapter_dir, dirs_exist_ok=True)

adapter_megabytes = (final_adapter_dir / "adapter_model.safetensors").stat().st_size / 1e6
print(f"converted     : {conversion['tensors']} tensors, rank {conversion['rank']}, alpha {conversion['lora_alpha']:g}")
print(f"adapters on   : {', '.join(conversion['target_modules'])}")
print(f"adapter saved : {final_adapter_dir}")
print(f"and copied to : {repo_adapter_dir}")
print(f"size          : {adapter_megabytes:.1f} MB")

**Why this cell:** do not take a converter's word for it. Pick one adapted layer, push the same random input through the MLX formula with the MLX matrices and through the PEFT formula with the converted matrices, and compare. The difference should be float16 rounding and nothing more.

In [ ]:
import numpy as np
from safetensors.numpy import load_file

mlx_weights = load_file(str(mlx_adapter_dir / finetune_mlx_utils.MLX_ADAPTER_FILE))
peft_weights = load_file(str(final_adapter_dir / "adapter_model.safetensors"))

layer = "model.layers.0.self_attn.q_proj" if "model.layers.0.self_attn.q_proj.lora_a" in mlx_weights else sorted(mlx_weights)[0].rsplit(".", 1)[0]
lora_a = mlx_weights[f"{layer}.lora_a"].astype("float32")
lora_b = mlx_weights[f"{layer}.lora_b"].astype("float32")
matrix_a = peft_weights[f"base_model.model.{layer}.lora_A.weight"].astype("float32")
matrix_b = peft_weights[f"base_model.model.{layer}.lora_B.weight"].astype("float32")

test_input = np.random.default_rng(SEED).normal(size=(4, lora_a.shape[0])).astype("float32")
mlx_output = lora_parameters["scale"] * (test_input @ lora_a) @ lora_b
peft_output = (conversion["lora_alpha"] / conversion["rank"]) * (test_input @ matrix_a.T) @ matrix_b.T

largest_difference = float(np.abs(mlx_output - peft_output).max())
largest_value = float(np.abs(mlx_output).max())
print(f"layer checked      : {layer}")
print(f"shapes  MLX  a, b  : {lora_a.shape}, {lora_b.shape}")
print(f"shapes  PEFT A, B  : {matrix_a.shape}, {matrix_b.shape}")
print(f"largest difference : {largest_difference:.2e}   (largest output value {largest_value:.2e})")
assert largest_difference <= 0.02 * largest_value + 1e-4, "the converted adapter does not compute the same numbers"
print("the converted adapter computes the same numbers")

**Why this cell (milestone 4) — the honest test:** the model in memory is not the deliverable; the files on disk are. So load a *fresh* base model with the saved adapter attached and ask the same preview tickets as before. Compare with the BEFORE replies: is `asset_tag` now `null` when the ticket names no asset? Is `routing_queue` a real queue?

In [ ]:
tuned_model, tuned_tokenizer = finetune_mlx_utils.load_tuned_model(model_choice["mlx_repo"], run_dir)

after_replies = []
after_valid_count = 0
for pair in preview_pairs:
    reply_text = finetune_mlx_utils.generate_reply(tuned_model, tuned_tokenizer, pair["messages"][:2], NEW_TOKENS)
    verdict = finetune_utils.check_reply(reply_text, schema)
    after_valid_count += verdict["valid"]
    after_replies.append({"ticket_id": pair["ticket_id"], "reply": reply_text, "valid": verdict["valid"]})
    print(f"{pair['ticket_id']}  schema-valid: {verdict['valid']}")
    print(f"  tuned    : {reply_text[:300]}")
    print(f"  expected : {dataset_utils.pair_completion_text(pair)}")

utils.save_json(run_dir, "replies_after", after_replies)
print()
print(f"BEFORE training: {before_valid_count} of {len(before_replies)} schema-valid")
print(f"AFTER  training: {after_valid_count} of {len(after_replies)} schema-valid   (adapter loaded back from disk)")

**Why this cell (optional, never blocks you):** everything after today reaches your tuned model through one name — the `tuned` endpoint in `config/endpoints.py`, an Ollama model called `oq-ticket-tuned`. `scripts/register_adapter.py` writes a two-line Modelfile (`FROM` the same base model, `ADAPTER` your *converted* folder) and runs `ollama create`. On a Mac with Ollama installed this works right here; if Ollama is not installed the cell says so and moves on.

In [ ]:
import subprocess

register_script = REPO_ROOT / "scripts" / "register_adapter.py"

if SMOKE_TEST:
    print("Smoke test: the tiny model has no Ollama twin, so there is nothing to register.")
elif shutil.which("ollama") is None:
    print("Ollama is not installed here - skipped. Notebook 06 registers the adapter.")
else:
    command = [sys.executable, str(register_script), "--adapter", str(repo_adapter_dir)]
    completed = subprocess.run(command, capture_output=True, text=True)
    print(completed.stdout + completed.stderr)
    if completed.returncode != 0:
        print("Not registered (see the message above). This does not affect your adapter - notebook 06 will try again.")

**Why this cell:** the declared result in one block. A handful of tickets is a taste, not a measurement — the real score comes in notebook 06, where the eval harness runs all 20 held-out tickets against base and tuned.

In [ ]:
summary = {
    "model": model_choice["hf_repo"],
    "trained_with": f"MLX ({model_choice['mlx_repo']})",
    "ollama_base": model_choice["ollama_base"],
    "run_folder": str(run_dir),
    "adapter_in_run_folder": str(final_adapter_dir),
    "adapter_in_repo": str(repo_adapter_dir),
    "adapter_format": "PEFT (converted from MLX)",
    "adapter_megabytes": round(adapter_megabytes, 1),
    "train_rows": len(train_set),
    "epochs_done": progress["epochs_done"],
    "epochs_planned": NUM_EPOCHS,
    "first_loss": first_loss,
    "last_loss": last_loss,
    "training_minutes": round(training_minutes, 1),
    "valid_before": before_valid_count,
    "valid_after": after_valid_count,
    "preview_tickets": len(preview_pairs),
}
utils.save_json(run_dir, "05b_summary", summary)

print()
print("ADAPTER READY" if not SMOKE_TEST else "SMOKE TEST COMPLETE (plumbing only - this tiny model has not learned the task)")
print(f"  base model        : {summary['model']}")
print(f"  trained           : {summary['epochs_done']} of {summary['epochs_planned']} epochs, "
      f"{summary['training_minutes']:.1f} minutes, MLX on {mx.default_device()}")
print(f"  training loss     : {summary['first_loss']:.3f} -> {summary['last_loss']:.3f}")
print(f"  schema-valid JSON : {summary['valid_before']} of {summary['preview_tickets']} before -> "
      f"{summary['valid_after']} of {summary['preview_tickets']} after")
print(f"  adapter           : {summary['adapter_megabytes']} MB, {summary['adapter_format']} - the same format notebook 05 produces")
print(f"    in the run folder : {summary['adapter_in_run_folder']}")
print(f"    in the repo       : {summary['adapter_in_repo']}")
print("  next              : notebook 06 scores it on all 20 held-out tickets against the base model")